# İzmir Public Services — EDA & Quality Checks
A reproducible starter notebook for the **Izmir Public Services Open Data Snapshot**. It verifies the dataset contract, summarizes the five service layers, and produces simple geospatial views without making travel-time or neighborhood-quality claims.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

matches = list(Path('/kaggle/input').rglob('izmir_public_services_combined.csv'))
if not matches:
    available = [str(p) for p in Path('/kaggle/input').rglob('*') if p.is_file()]
    raise FileNotFoundError(f'Combined CSV not mounted. Available files: {available[:50]}')
DATA = matches[0]
print(f'Data file: {DATA}')
df = pd.read_csv(DATA)
print(f'Rows: {len(df):,}')
print(f'Service types: {df.service_type.nunique()}')
df.head()


## Contract checks
These checks are intentionally simple and transparent: unique IDs, valid coordinates, non-empty names, and a single declared source license.


In [ ]:
checks = {
    'unique_record_id': df['record_id'].is_unique,
    'latitude_valid': df['latitude'].between(-90, 90).all(),
    'longitude_valid': df['longitude'].between(-180, 180).all(),
    'name_present': df['name'].fillna('').str.strip().ne('').all(),
    'license_cc_by_4': set(df['source_license'].dropna()) == {'CC BY 4.0'},
}
pd.Series(checks, name='PASS')


## Service-layer composition


In [ ]:
counts = df['service_type_label'].value_counts().sort_values()
ax = counts.plot(kind='barh', figsize=(9, 4), title='Records by public-service layer')
ax.set_xlabel('Records')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
counts.to_frame('records')


## District coverage
District fields are source-dependent. Blank district values should be treated as **metadata not supplied**, not as a separate district and not as evidence of service absence.


In [ ]:
district = (df.assign(district_clean=df['district'].fillna('').str.strip())
              .query("district_clean != ''")
              .groupby(['district_clean','service_type_label'])
              .size().unstack(fill_value=0))
district['total'] = district.sum(axis=1)
district.sort_values('total', ascending=False).head(15)


## Geospatial overview
This is a point distribution only. It is **not** a route map or a travel-time accessibility model.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for label, g in df.groupby('service_type_label'):
    ax.scatter(g['longitude'], g['latitude'], s=8, alpha=.55, label=label)
ax.set_title('İzmir public-service point distribution')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()


## Responsible-use boundary
- A missing record does not mean a service does not exist.
- Straight-line proximity is not walking, driving, transit, accessibility, or emergency-response time.
- Duty-pharmacy data is time-sensitive; inspect `observed_at` and `retrieved_at`.
- No personal user location is included in this dataset.

Source attribution: **İzmir Büyükşehir Belediyesi Açık Veri Portalı — CC BY 4.0**.
